# scanpy

## Example 1: QC metrics and normalisation

In [ ]:
# Example 1: calculate_qc_metrics, normalize_total and log1p on synthetic counts
import numpy as np
import anndata as ad
import scanpy as sc

rng = np.random.default_rng(0)
adata = ad.AnnData(rng.poisson(1.0, size=(300, 50)).astype(np.float32))
sc.pp.calculate_qc_metrics(adata, percent_top=(10, 20), inplace=True)
assert {"n_genes_by_counts", "total_counts"} <= set(adata.obs.columns)
sc.pp.normalize_total(adata, target_sum=1e4)
assert np.allclose(adata.X.sum(axis=1), 1e4, rtol=1e-4)
sc.pp.log1p(adata)
print("Example 1:", adata.shape, float(adata.X.max()))

## Example 2: highly variable genes and PCA

In [ ]:
# Example 2: highly_variable_genes then pca (sklearn underneath); PC1 splits two planted groups
import numpy as np
import anndata as ad
import scanpy as sc

rng = np.random.default_rng(1)
counts = rng.poisson(1.0, size=(200, 100)).astype(np.float32)
counts[:100, :10] += 20  # a block of genes that vary between two groups
adata = ad.AnnData(counts)
sc.pp.normalize_total(adata); sc.pp.log1p(adata)
sc.pp.highly_variable_genes(adata, n_top_genes=20)
sc.pp.pca(adata, n_comps=10)
assert adata.var["highly_variable"].sum() == 20 and adata.obsm["X_pca"].shape == (200, 10)
pc1 = adata.obsm["X_pca"][:, 0]
assert abs(pc1[:100].mean() - pc1[100:].mean()) > 3 * pc1[100:].std()  # PC1 separates the planted groups
print("Example 2:", adata.obsm["X_pca"].shape, adata.uns["pca"]["variance_ratio"][:3].round(3))

## Example 3: neighbours and UMAP

In [ ]:
# Example 3: neighbors (pynndescent / umap) then tl.umap separates two planted groups
import numpy as np
import anndata as ad
import scanpy as sc

rng = np.random.default_rng(2)
counts = rng.poisson(1.0, size=(200, 60)).astype(np.float32)
counts[:100, :15] += 15
adata = ad.AnnData(counts)
sc.pp.normalize_total(adata); sc.pp.log1p(adata); sc.pp.pca(adata, n_comps=10)
sc.pp.neighbors(adata, n_neighbors=10, random_state=0)
sc.tl.umap(adata, random_state=0)
emb = adata.obsm["X_umap"]
assert emb.shape == (200, 2) and np.linalg.norm(emb[:100].mean(0) - emb[100:].mean(0)) > 2.0
print("Example 3:", emb.shape, "connectivities", adata.obsp["connectivities"].nnz)

## Example 4: rank genes between groups

In [ ]:
# Example 4: rank_genes_groups (t-test) finds the planted marker genes of group "a"
import numpy as np
import anndata as ad
import scanpy as sc

rng = np.random.default_rng(3)
counts = rng.poisson(1.0, size=(160, 40)).astype(np.float32)
counts[:80, :5] += 10
adata = ad.AnnData(counts)
adata.obs["group"] = ["a"] * 80 + ["b"] * 80
adata.obs["group"] = adata.obs["group"].astype("category")
sc.pp.normalize_total(adata); sc.pp.log1p(adata)
sc.tl.rank_genes_groups(adata, groupby="group", method="t-test")
top = [str(name) for name in adata.uns["rank_genes_groups"]["names"]["a"][:5]]
assert set(top) == {"0", "1", "2", "3", "4"}, top
print("Example 4: top markers of group a:", top)

## Example 5: plotting to a file

In [ ]:
# Example 5: pl.umap colours the embedding by a column and saves a PNG (headless matplotlib)
import pathlib, tempfile
import matplotlib
matplotlib.use("Agg")
import numpy as np
import anndata as ad
import scanpy as sc

rng = np.random.default_rng(4)
adata = ad.AnnData(rng.poisson(1.0, size=(120, 30)).astype(np.float32))
adata.obs["half"] = (["x"] * 60 + ["y"] * 60)
sc.pp.normalize_total(adata); sc.pp.log1p(adata); sc.pp.pca(adata, n_comps=5)
sc.pp.neighbors(adata, n_neighbors=8); sc.tl.umap(adata, random_state=0)
out = pathlib.Path(tempfile.mkdtemp()) / "umap.png"
fig = sc.pl.umap(adata, color="half", show=False, return_fig=True)
fig.savefig(out)
assert out.read_bytes()[:8] == b"\x89PNG\r\n\x1a\n"
print("Example 5:", out.name, out.stat().st_size, "bytes")